In [2]:
import sys
# Tell Python to look in your D: drive folder first
sys.path.insert(0, "D:\\PythonLibs") 

# Now try the imports
import torch
import torchvision
print("Torchvision version:", torchvision.__version__)

Torchvision version: 0.20.1+cu121


In [3]:
import numpy as np
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import rasterio
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import (
    PeakSignalNoiseRatio,
    StructuralSimilarityIndexMeasure,
)
from tqdm.notebook import tqdm

In [4]:
import torch

print(f"Is CUDA available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: PyTorch is using the CPU. The CUDA installation failed.")

Is CUDA available? True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM: 6.44 GB


In [5]:
# ============================================================
# 1.  DUAL-BRANCH FEATURE EXTRACTION + GATED FUSION
# ============================================================

class DoubleConv(nn.Module):
    """Standard double convolution block (Conv-BN-ReLU x2)."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class SAR_Prior_UNet(nn.Module):
    """
    U-Net that translates SAR (S1, 2ch) into a feature map f_sar (hidden_dim ch)
    and an auxiliary simulated optical image sim_opt (out_channels ch).

    Returns: (f_sar, sim_opt)
    """
    def __init__(self, in_channels=2, out_channels=13, hidden_dim=64):
        super().__init__()

        # Encoder
        self.inc   = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64,  128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder
        self.up1   = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        self.up2   = nn.ConvTranspose2d(512,  256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512,  256)
        self.up3   = nn.ConvTranspose2d(256,  128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256,  128)
        self.up4   = nn.ConvTranspose2d(128,   64, kernel_size=2, stride=2)
        # Penultimate hidden representation
        self.conv4 = DoubleConv(128, hidden_dim)

        # FIX 1+2: auxiliary output head — was defined but never called; forward
        # now returns (f_sar, sim_opt) instead of just f_sar
        self.outc  = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # Encode
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decode
        x = self.conv1(torch.cat([self.up1(x5), x4], dim=1))
        x = self.conv2(torch.cat([self.up2(x),  x3], dim=1))
        x = self.conv3(torch.cat([self.up3(x),  x2], dim=1))
        x = torch.cat([self.up4(x), x1], dim=1)

        f_sar   = self.conv4(x)          # hidden feature map
        sim_opt = self.outc(f_sar)       # FIX 1: actually call the output head

        return f_sar, sim_opt            # FIX 2: return both


class Cloudy_Optical_UNet(nn.Module):
    """
    U-Net encoder-decoder for the cloudy S2 image.
    Returns f_c (hidden_dim ch) — the cloudy optical feature map.
    """
    def __init__(self, in_channels=13, hidden_dim=64):
        super().__init__()

        # Encoder
        self.inc   = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64,  128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder
        self.up1   = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        self.up2   = nn.ConvTranspose2d(512,  256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512,  256)
        self.up3   = nn.ConvTranspose2d(256,  128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256,  128)
        self.up4   = nn.ConvTranspose2d(128,   64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, hidden_dim)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.conv1(torch.cat([self.up1(x5), x4], dim=1))
        x = self.conv2(torch.cat([self.up2(x),  x3], dim=1))
        x = self.conv3(torch.cat([self.up3(x),  x2], dim=1))
        x = torch.cat([self.up4(x), x1], dim=1)

        f_c = self.conv4(x)
        return f_c


class CrossModalGatedFusion(nn.Module):
    """
    FIX 3: This class was referenced in DualBranchCloudRemoval but never defined.

    Gated cross-modal fusion of SAR features (f_sar) and cloudy optical
    features (f_c), both of shape (B, dim, H, W).

    A soft gate g ∈ [0,1] is learned per spatial location:
        fused = g * f_sar + (1 - g) * f_c

    The gate is conditioned on the concatenation of both feature maps so the
    network can learn where to trust the SAR prior vs the optical signal.
    """
    def __init__(self, dim: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(dim * 2, dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.Sigmoid(),                # gate values in [0, 1]
        )
        self.norm = nn.BatchNorm2d(dim)

    def forward(self, f_sar: torch.Tensor, f_c: torch.Tensor) -> torch.Tensor:
        g      = self.gate(torch.cat([f_sar, f_c], dim=1))
        fused  = g * f_sar + (1.0 - g) * f_c
        return self.norm(fused)


class DualBranchCloudRemoval(nn.Module):
    """
    Stage-1 model:
      SAR U-Net  +  Optical U-Net  →  CrossModalGatedFusion  →  reconstruction head

    Returns (clear_reconstructed_img, sim_opt) so training can apply an
    auxiliary loss on the SAR branch output.
    """
    def __init__(self, sar_channels=2, opt_channels=13, hidden_dim=64):
        super().__init__()

        self.sar_branch     = SAR_Prior_UNet(in_channels=sar_channels,
                                             out_channels=opt_channels,
                                             hidden_dim=hidden_dim)
        self.optical_branch = Cloudy_Optical_UNet(in_channels=opt_channels,
                                                  hidden_dim=hidden_dim)

        # FIX 3: CrossModalGatedFusion now properly defined above
        self.fusion         = CrossModalGatedFusion(dim=hidden_dim)

        self.reconstruction_head = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim // 2, opt_channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, sar_img: torch.Tensor, cloudy_opt_img: torch.Tensor):
        # FIX 4: sar_branch now returns (f_sar, sim_opt) — unpack correctly
        f_sar, sim_opt      = self.sar_branch(sar_img)
        f_c                 = self.optical_branch(cloudy_opt_img)
        fused               = self.fusion(f_sar, f_c)
        clear_reconstructed = self.reconstruction_head(fused)

        # Also expose (fused, f_c) so FormerCR_Core can refine further
        return clear_reconstructed, sim_opt, fused, f_c

In [6]:
# ============================================================
# 3.  DATASET
# ============================================================

class SEN12MSCR_Dataset(Dataset):
    """
    Loads matched (S1, S2-cloudy, S2-clear) triplets.

    root_dir layout:
        root_dir/
          ROIs1158_spring / s1 / *.tif
                          / s2_cloudy / *.tif
                          / s2 / *.tif
          ROIs1868_summer / ...
          ROIs1970_fall   / ...
          ROIs2017_winter / ...
    """

    SEASONS = [
        "ROIs1158_spring",
        "ROIs1868_summer",
        "ROIs1970_fall",
        "ROIs2017_winter",
    ]

    def __init__(self, root_dir: str, split: str = "train"):
        if split not in ("train", "val", "test"):
            raise ValueError("split must be 'train', 'val', or 'test'.")

        self.split    = split
        self.triplets: list = []

        def get_id(fname: str):
            m = re.search(r"(\d+_[pP]\d+)", fname)
            return m.group(1) if m else None

        for season in self.SEASONS:
            season_dir = os.path.join(root_dir, season)
            if not os.path.exists(season_dir):
                print(f"Warning: {season_dir} not found — skipping.")
                continue

            s1_dir  = os.path.join(season_dir, "s1")
            s2c_dir = os.path.join(season_dir, "s2_cloudy")
            s2_dir  = os.path.join(season_dir, "s2")

            s1_map  = {get_id(f): f for f in os.listdir(s1_dir)  if f.endswith(".tif")}
            s2c_map = {get_id(f): f for f in os.listdir(s2c_dir) if f.endswith(".tif")}
            s2_map  = {get_id(f): f for f in os.listdir(s2_dir)  if f.endswith(".tif")}

            common_ids = set(s1_map) & set(s2c_map) & set(s2_map)
            common_ids.discard(None)
            sorted_ids = sorted(common_ids)

            if split == "train":
                split_ids = sorted_ids[:2500]
            elif split == "val":
                split_ids = sorted_ids[2500:3000]
            else:
                split_ids = sorted_ids[3000:3500]

            for uid in split_ids:
                self.triplets.append({
                    "s1":       os.path.join(s1_dir,  s1_map[uid]),
                    "s2c":      os.path.join(s2c_dir, s2c_map[uid]),
                    "s2_clear": os.path.join(s2_dir,  s2_map[uid]),
                })

        print(f"Initialised '{self.split}' dataset with {len(self.triplets)} images.")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        paths = self.triplets[idx]


        # Read and cast natively in numpy to prevent 64-bit RAM bloat

        with rasterio.open(paths["s1"]) as src:
            s1 = torch.from_numpy(src.read().astype(np.float32))
        with rasterio.open(paths["s2c"]) as src:
            s2c = torch.from_numpy(src.read().astype(np.float32))
        with rasterio.open(paths["s2_clear"]) as src:
            s2_clear = torch.from_numpy(src.read().astype(np.float32))

        # Clamp and scale
        s1       = torch.clamp((s1 + 25.0) / 25.0,  0.0, 1.0)
        s2c      = torch.clamp(s2c      / 10000.0,   0.0, 1.0)
        s2_clear = torch.clamp(s2_clear / 10000.0,   0.0, 1.0)

        return s1, s2c, s2_clear




In [7]:

# ============================================================
# 4.  VISUALISATION
# ============================================================

def visualize_results(dual_branch, former_cr, dataloader, device,
                      epoch, phase, num_examples=5, save_dir="visuals"):
    os.makedirs(save_dir, exist_ok=True)
    dual_branch.eval()
    former_cr.eval()

    fig, axes = plt.subplots(num_examples, 3, figsize=(12, 4 * num_examples))
    axes[0, 0].set_title("Cloudy Input (RGB proxy)")
    axes[0, 1].set_title("Reconstructed Clear (RGB)")
    axes[0, 2].set_title("Ground Truth Clear (RGB)")

    count = 0
    with torch.no_grad():
        for s1, s2c, s2_clear in dataloader:
            s1       = s1.to(device)
            s2c      = s2c.to(device)
            s2_clear = s2_clear.to(device)

            # FIX 9: call dual_branch (not old CrossModalFusion stem)
            _, _, fused, f_c = dual_branch(s1, s2c)
            outputs = former_cr(fused, f_c)

            for i in range(outputs.size(0)):
                if count >= num_examples:
                    break

                rgb_out      = outputs[i, [3, 2, 1]].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
                rgb_gt       = s2_clear[i, [3, 2, 1]].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
                cloudy_proxy = s2c[i].mean(dim=0).cpu().numpy()

                axes[count, 0].imshow(cloudy_proxy, cmap="gray")
                axes[count, 1].imshow(rgb_out)
                axes[count, 2].imshow(rgb_gt)
                for ax in axes[count]:
                    ax.axis("off")
                count += 1

            if count >= num_examples:
                break

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"epoch_{epoch}_{phase}.png")
    plt.savefig(save_path)
    plt.close()
    print(f"  Saved visualisation → {save_path}")

In [10]:
# ============================================================
# 2.  FORMER-CR REFINEMENT BACKBONE
# ============================================================

class WindowAttention(nn.Module):
    """Window-based Multi-Head Self-Attention (W-MSA)."""

    def __init__(self, dim, window_size=8, num_heads=4):
        super().__init__()
        self.dim         = dim
        self.window_size = window_size
        self.num_heads   = num_heads
        self.attn        = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads,
                                                  batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape

        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x = F.pad(x, (0, pad_r, 0, pad_b))

        _, _, Hp, Wp = x.shape
        ws = self.window_size

        x       = x.view(B, C, Hp // ws, ws, Wp // ws, ws)
        x       = x.permute(0, 2, 4, 3, 5, 1).contiguous()
        windows = x.view(-1, ws * ws, C)

        # Cast to float32 for stable attention computation
        windows = windows.float()
        attn_windows, _ = self.attn(windows, windows, windows)
        attn_windows = attn_windows.to(x.dtype)

        attn_windows = attn_windows.view(B, Hp // ws, Wp // ws, ws, ws, C)
        x            = attn_windows.permute(0, 5, 1, 3, 2, 4).contiguous()
        x            = x.view(B, C, Hp, Wp)

        if pad_r > 0 or pad_b > 0:
            x = x[:, :, :H, :W].contiguous()

        return x


class LeFF(nn.Module):
    """Locally-enhanced Feed-Forward Network."""

    def __init__(self, dim, hidden_ratio=4):
        super().__init__()
        hidden_dim  = int(dim * hidden_ratio)
        self.proj1  = nn.Conv2d(dim, hidden_dim, kernel_size=1)
        self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1,
                                groups=hidden_dim)
        self.act    = nn.GELU()
        self.proj2  = nn.Conv2d(hidden_dim, dim, kernel_size=1)

    def forward(self, x):
        return self.proj2(self.act(self.dwconv(self.proj1(x))))


class LewinTransformerLayer(nn.Module):
    def __init__(self, dim, window_size=8, num_heads=4):
        super().__init__()
        self.norm1 = nn.GroupNorm(1, dim)
        self.attn  = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.GroupNorm(1, dim)
        self.leff  = LeFF(dim)

    def forward(self, x):
        # Cast to float32 for GroupNorm stability, then cast back
        x = x + self.attn(self.norm1(x.float()).to(x.dtype))
        x = x + self.leff(self.norm2(x.float()).to(x.dtype))
        return x


class LTDS(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.down = nn.Conv2d(in_dim, out_dim, kernel_size=4, stride=2, padding=1)
        self.ltl  = LewinTransformerLayer(out_dim)

    def forward(self, x):
        return self.ltl(self.down(x))


class LTUS(nn.Module):
    def __init__(self, in_dim, skip_dim, out_dim):
        super().__init__()
        self.up_conv       = nn.Conv2d(in_dim, (in_dim // 2) * 4, kernel_size=1)
        self.pixel_shuffle = nn.PixelShuffle(2)
        self.reduce        = nn.Conv2d((in_dim // 2) + skip_dim, out_dim, kernel_size=1)
        self.ltl           = LewinTransformerLayer(out_dim)

    def forward(self, x, skip):
        x = self.pixel_shuffle(self.up_conv(x))
        x = self.reduce(torch.cat([x, skip], dim=1))
        return self.ltl(x)


class SEDecoder(nn.Module):
    def __init__(self, in_dim, out_channels=13):
        super().__init__()
        r = 4
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1         = nn.Conv2d(in_dim, in_dim // r, kernel_size=1)
        self.relu        = nn.ReLU(inplace=True)
        self.fc2         = nn.Conv2d(in_dim // r, in_dim, kernel_size=1)
        self.sigmoid_se  = nn.Sigmoid()
        self.reconstruct = nn.Sequential(
            nn.Conv2d(in_dim, out_channels, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        scale = self.sigmoid_se(self.fc2(self.relu(self.fc1(self.global_pool(x)))))
        return self.reconstruct(x * scale)


class FormerCR_Core(nn.Module):
    """
    Lewin-Transformer U-Net that refines the dual-branch fused features.

    Inputs:
        fused_features  – output of CrossModalGatedFusion  (B, base_dim, H, W)
        cloudy_features – f_c from Cloudy_Optical_UNet     (B, base_dim, H, W)
    """
    def __init__(self, base_dim=64, out_channels=13):
        super().__init__()

        self.ltds1 = LTDS(base_dim,      base_dim * 2)
        self.ltds2 = LTDS(base_dim * 2,  base_dim * 4)
        self.ltds3 = LTDS(base_dim * 4,  base_dim * 8)
        self.ltds4 = LTDS(base_dim * 8,  base_dim * 16)

        self.bottleneck = nn.Sequential(
            LewinTransformerLayer(base_dim * 16),
            LewinTransformerLayer(base_dim * 16),
        )

        self.ltus4 = LTUS(base_dim * 16, base_dim * 8,  base_dim * 8)
        self.ltus3 = LTUS(base_dim * 8,  base_dim * 4,  base_dim * 4)
        self.ltus2 = LTUS(base_dim * 4,  base_dim * 2,  base_dim * 2)
        self.ltus1 = LTUS(base_dim * 2,  base_dim,      base_dim)

        self.se_decoder = SEDecoder(base_dim, out_channels)

    def forward(self, fused_features, cloudy_features):
        skip1 = fused_features
        skip2 = self.ltds1(skip1)
        skip3 = self.ltds2(skip2)
        skip4 = self.ltds3(skip3)
        x     = self.ltds4(skip4)

        x = self.bottleneck(x)
        x = self.ltus4(x, skip4)
        x = self.ltus3(x, skip3)
        x = self.ltus2(x, skip2)
        x = self.ltus1(x, skip1)

        x = x + cloudy_features          # residual branch
        return self.se_decoder(x)


# ============================================================
# 5.  TRAINING LOOP
# ============================================================
import os

def train_former_cr(dual_branch, former_cr, train_loader, val_loader,
                    device, epochs=50, lr=1e-4, aux_weight=0.1, resume_ckpt=None):
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(
        list(dual_branch.parameters()) + list(former_cr.parameters()),
        lr=lr, weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    dual_branch = dual_branch.to(device)
    former_cr   = former_cr.to(device)

    scaler = torch.amp.GradScaler(device.type)

    start_epoch = 1

    # ── RESUME FROM CHECKPOINT ────────────────────────────────
    if resume_ckpt and os.path.exists(resume_ckpt):
        print(f"\n[*] Found checkpoint! Loading weights from {resume_ckpt}...")
        ckpt = torch.load(resume_ckpt, map_location=device)
        dual_branch.load_state_dict(ckpt["dual_branch"])
        former_cr.load_state_dict(ckpt["former_cr"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1

        # Fast-forward the scheduler to match the resumed epoch
        for _ in range(start_epoch - 1):
            scheduler.step()

        print(f"[*] Successfully resumed. Starting training from Epoch {start_epoch}\n")
    else:
        print("\n[*] No checkpoint found or provided. Starting from Epoch 1.\n")

    for epoch in range(start_epoch, epochs + 1):
        # ── TRAINING ──────────────────────────────────────────
        dual_branch.train()
        former_cr.train()
        train_loss = 0.0

        for batch_idx, (s1, s2c, s2_clear) in enumerate(train_loader):
            s1, s2c, s2_clear = s1.to(device), s2c.to(device), s2_clear.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type=device.type):
                clear_stage1, sim_opt, fused, f_c = dual_branch(s1, s2c)
                clear_stage2 = former_cr(fused, f_c)

                main_loss = criterion(clear_stage2, s2_clear)
                aux_loss  = criterion(sim_opt,      s2_clear)
                loss      = main_loss + aux_weight * aux_loss
                
            
                if torch.isnan(loss):
                    print(f"!!! NaN detected at Batch {batch_idx}. Cleaning memory and skipping...")
                    
                    # 1. Clear the graph and delete the tensor to free VRAM
                    del loss, main_loss, aux_loss, clear_stage1, sim_opt, fused, f_c, clear_stage2
                    
                    # 2. Force an immediate cleanup of the memory
                    torch.cuda.empty_cache() 
                    
                    # 3. Reset gradients for the next healthy batch
                    optimizer.zero_grad()
                    
                    # 4. Move to next batch without triggering backprop
                    continue 

            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                list(dual_branch.parameters()) + list(former_cr.parameters()),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

            
            print(f"Epoch [{epoch}/{epochs}] Batch [{batch_idx}/{len(train_loader)}] | Batch Loss: {loss.item():.4f}")

        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch [{epoch}/{epochs}] | Train Loss: {avg_train_loss:.4f}")

        # ── SAVE CHECKPOINT EVERY EPOCH ───────────────────────
        ckpt_dict = {
            "epoch":       epoch,
            "dual_branch": dual_branch.state_dict(),
            "former_cr":   former_cr.state_dict(),
            "optimizer":   optimizer.state_dict(),
        }

        epoch_ckpt_path  = f"/kaggle/working/former_cr_epoch_{epoch}.pth"
        latest_ckpt_path = "/kaggle/working/former_cr_latest.pth"

        torch.save(ckpt_dict, epoch_ckpt_path)
        torch.save(ckpt_dict, latest_ckpt_path)
        print(f"[*] Checkpoint saved to {epoch_ckpt_path}")

        # ── PERIODIC VISUALISATION (every 5 epochs) ───────────
        if epoch % 5 == 0 or epoch == 1:
            visualize_results(dual_branch, former_cr, train_loader, device, epoch, "train")
            visualize_results(dual_branch, former_cr, val_loader,   device, epoch, "val")

        # ── VALIDATION (every 5 epochs) ───────────────────────
        if epoch % 5 == 0:
            dual_branch.eval()
            former_cr.eval()
            val_loss = 0.0
            psnr_metric.reset()
            ssim_metric.reset()

            with torch.no_grad():
                with torch.amp.autocast(device_type=device.type):
                    for s1, s2c, s2_clear in val_loader:
                        s1       = s1.to(device)
                        s2c      = s2c.to(device)
                        s2_clear = s2_clear.to(device)

                        _, sim_opt, fused, f_c = dual_branch(s1, s2c)
                        outputs = former_cr(fused, f_c)

                        val_loss += (criterion(outputs, s2_clear)
                                     + aux_weight * criterion(sim_opt, s2_clear)).item()
                        psnr_metric.update(outputs, s2_clear)
                        ssim_metric.update(outputs, s2_clear)

            avg_val_loss = val_loss / len(val_loader)
            val_psnr     = psnr_metric.compute().item()
            val_ssim     = ssim_metric.compute().item()

            print(f"--- Eval Epoch {epoch} ---")
            print(f"Val Loss : {avg_val_loss:.4f}")
            print(f"Val PSNR : {val_psnr:.2f} dB")
            print(f"Val SSIM : {val_ssim:.4f}")
            print("-" * 25)


# ============================================================
# 6.  ENTRY POINT
# ============================================================

if __name__ == "__main__":
    DATA_ROOT  = "D:\extracted_data"
    EPOCHS     = 50
    LR         = 2e-4
    BATCH      = 4
    AUX_WEIGHT = 0.1

    # Always resume from latest — auto-updated every epoch
    CHECKPOINT_PATH = "D:\extracted_data\former_cr_epoch_27.pth"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_ds = SEN12MSCR_Dataset(DATA_ROOT, split="train")
    val_ds   = SEN12MSCR_Dataset(DATA_ROOT, split="val")

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                              num_workers=0, pin_memory=True)

    dual_branch = DualBranchCloudRemoval(sar_channels=2, opt_channels=13,
                                         hidden_dim=64).to(device)
    former_cr   = FormerCR_Core(base_dim=64, out_channels=13).to(device)

    # Force MultiheadAttention and GroupNorm to float32 for numerical stability
    for module in former_cr.modules():
        if isinstance(module, (nn.MultiheadAttention, nn.GroupNorm)):
            module.to(torch.float32)

    train_former_cr(dual_branch, former_cr, train_loader, val_loader,
                    device, epochs=EPOCHS, lr=LR, aux_weight=AUX_WEIGHT,
                    resume_ckpt=CHECKPOINT_PATH)

Using device: cuda
Initialised 'train' dataset with 9544 images.
Initialised 'val' dataset with 913 images.

[*] No checkpoint found or provided. Starting from Epoch 1.

Epoch [1/50] Batch [0/2386] | Batch Loss: 0.4074


KeyboardInterrupt: 

In [ ]:
!find /kaggle/input/ -name "*.pth"

In [ ]:
import os
import shutil

# --- 1. CLEANUP CHECKPOINTS ---
working_dir = "/kaggle/working/"
keep_epoch = "former_cr_epoch_28.pth"

print("--- Cleaning Checkpoints ---")
for filename in os.listdir(working_dir):
    # Only delete files that start with 'former_cr' and end with '.pth'
    if filename.startswith("former_cr") and filename.endswith(".pth"):
        # Keep the 28th epoch, and the 'latest' file
        if filename != keep_epoch :
            os.remove(os.path.join(working_dir, filename))
            print(f"Deleted: {filename}")
        

# --- 2. CLEANUP VISUALS ---
print("\n--- Cleaning Visuals ---")
visuals_dir = os.path.join(working_dir, "visuals")
if os.path.exists(visuals_dir):
    # This deletes the folder and everything inside it
    shutil.rmtree(visuals_dir)
    # Recreate the folder empty
    os.makedirs(visuals_dir)
    print("Deleted all files in /visuals/ and recreated empty folder.")

# --- 3. FINAL STATUS ---
print(f"\nCleanup complete. Remaining space: {shutil.disk_usage(working_dir).free / 1e9:.2f} GB free.")